# NC Türkçe BPE · Colab'da eğitim

1. Kurulum
2. Yayımlanan tokenizer'ların bu kodla yeniden kurulması (doğrulama)
3. Eğitim: yayımlanan ayarlarla ya da kendi ayarlarınızla
4. Kontrol ve örnek

CPU çalışma zamanı yeterli. Sayım ve Zemberek sonuçları Drive'daki önbelleğe yazılır; aynı derlemle sonraki eğitimler bu adımları atlar.

In [ ]:
# 1 · Kurulum (yeni bir çalışma zamanında bir kez; Colab yeniden başlatma isterse başlatıp bu hücreyi atlayın)
!pip install -q "nc-bpe @ git+https://github.com/nesicn/nc-turkce-bpe.git"

In [ ]:
# 2 · Doğrulama: yayımlanan tokenizer.json'lar bu paketin iskeletiyle byte byte aynı kuruluyor mu?
from huggingface_hub import hf_hub_download
from nc_bpe.egitim import yeniden_kur

for boyut in ("32k", "64k", "128k"):
    repo = f"nesicn/nc-turkce-bpe-{boyut}"
    yol = hf_hub_download(repo, "tokenizer.json")
    hf_hub_download(repo, "tokenizer_config.json")
    s = yeniden_kur(yol)
    print(f"{repo}: tokenizer.json {'aynı' if s.get('tokenizer.json') else 'FARKLI'} · "
          f"tokenizer_config.json {'aynı' if s.get('tokenizer_config.json') else 'FARKLI'} · sözlük {s.get('sozluk', 0):,}")

In [ ]:
# 3 · Derlem ve çıktı klasörleri
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")
DERLEM_DRIVE = "/content/drive/MyDrive/nc_derlem"          # .parquet / .jsonl / .txt dosyalarının klasörü
KOK = "/content/drive/MyDrive/nc_ciktilar"                  # tokenizer'lar ve önbellek
DERLEM = "/content/derlem"                                  # yerel diskten okumak Drive'dan okumaktan hızlı
if not os.path.isdir(DERLEM):
    shutil.copytree(DERLEM_DRIVE, DERLEM)
print(sum(os.path.getsize(os.path.join(k, a)) for k, _, ad in os.walk(DERLEM) for a in ad) / 1e6, "MB")

In [ ]:
# 4 · Eğitim
from nc_bpe.egitim import egit
from nc_bpe.tarif import HAZIR_AYARLAR

AD, SOZLUK, V1 = "nc-64k", HAZIR_AYARLAR["64k"]["sozluk"], HAZIR_AYARLAR["64k"]["v1"]   # deney için değiştirin
bilgi = egit(DERLEM, os.path.join(KOK, AD), SOZLUK, V1, onbellek=os.path.join(KOK, "onbellek"))
assert not bilgi["hatalar"], bilgi["hatalar"]

In [ ]:
# 5 · Kontrol ve örnek
import json
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(os.path.join(KOK, AD))
print(len(tok), bilgi["tokenizer_json_sha256"])
print(json.dumps(bilgi["egitim"], ensure_ascii=False))
for m in ["Ankara'da 1923'te Cumhuriyet ilan edildi.", "Kitaplarımızdan birini öğretmenlerimize verdik."]:
    ofs = tok(m, return_offsets_mapping=True)["offset_mapping"]
    print(" | ".join(m[a:b] for a, b in ofs))
    assert tok.decode(tok(m)["input_ids"]) == m